# 05_sensitivity_and_figures

Robustness of the null across BMI-reduction thresholds and definitions (Table 4), and all grayscale figures (600 dpi, PNG + PDF, no captions).

In [1]:
# 05_sensitivity_and_figures.ipynb
# Sensitivity grid for the null result and all publication figures.

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
FIG  = os.path.join(ROOT, "results", "figures")
TAB  = os.path.join(ROOT, "results", "tables")

# Grayscale theme
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "grid.color": "0.85",
                     "axes.grid": True, "savefig.dpi": 600, "figure.dpi": 120})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG, name + ".pdf"), bbox_inches="tight")
    plt.close(fig)

In [2]:
# Sensitivity grid: absolute and relative BMI-reduction thresholds (Table 4).
panel = pd.read_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
adult = panel[panel.age >= 19].copy()
TRIPLES = [(2019,2020,2021),(2020,2021,2022),(2021,2022,2023),(2022,2023,2024)]

def build(thr_type, thr):
    rows = []
    for t0, t1, t2 in TRIPLES:
        d0 = adult[adult.year==t0][["PIDWON","BMI","HTN","age","SEX","smoke_cur","exer_reg"]]
        d1 = adult[adult.year==t1][["PIDWON","BMI","HTN"]]
        d2 = adult[adult.year==t2][["PIDWON","HTN"]]
        m = (d0.merge(d1,on="PIDWON",suffixes=("_0","_1"))
               .merge(d2,on="PIDWON").rename(columns={"HTN":"HTN_2"}))
        r = m[(m["HTN_0"]==0)&(m["HTN_1"]==0)].dropna(subset=["BMI_0","BMI_1","HTN_2"]).copy()
        r["dBMI"] = r["BMI_1"] - r["BMI_0"]
        r["incident_t2"] = (r["HTN_2"]==1).astype(int)
        if thr_type == "abs":
            r["achieved"] = (r["dBMI"] <= -thr).astype(int)
        else:
            r["achieved"] = (r["dBMI"]/r["BMI_0"] <= -thr).astype(int)
        r["t0y"] = t0
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

def adj_or(S):
    S = S.copy(); S["female"] = (S["SEX"]==2).astype(int)
    S = S.dropna(subset=["smoke_cur","exer_reg","BMI_0","age"])
    m = smf.logit("incident_t2 ~ achieved + BMI_0 + age + female + smoke_cur + exer_reg + C(t0y)",
                  data=S).fit(disp=0)
    OR = np.exp(m.params["achieved"]); ci = np.exp(m.conf_int().loc["achieved"])
    return OR, ci[0], ci[1], m.pvalues["achieved"], int(S["achieved"].sum())

CONFIGS = [("abs",0.5,"Abs 0.5"),("abs",1.0,"Abs 1.0"),("abs",2.0,"Abs 2.0"),
           ("rel",0.03,"Rel 3%"),("rel",0.05,"Rel 5%"),("rel",0.07,"Rel 7%")]
recs = []
for tt, th, lab in CONFIGS:
    OR, lo, hi, p, n = adj_or(build(tt, th))
    recs.append({"Threshold":lab,"Achieved_N":n,"OR":round(OR,2),
                 "CI_low":round(lo,2),"CI_high":round(hi,2),"p":round(p,2)})
tab4 = pd.DataFrame(recs)
tab4.to_csv(os.path.join(TAB, "table4_sensitivity.csv"), index=False)
print(tab4.to_string(index=False))

Threshold  Achieved_N   OR  CI_low  CI_high    p
  Abs 0.5        4231 1.16    0.96     1.40 0.13
  Abs 1.0        2343 1.06    0.83     1.35 0.66
  Abs 2.0         809 0.84    0.56     1.27 0.41
   Rel 3%        3433 1.15    0.94     1.41 0.18
   Rel 5%        1859 1.06    0.81     1.39 0.66
   Rel 7%        1096 0.84    0.58     1.21 0.34


In [3]:
# Figure 1: incidence rate by transition interval.
inter  = ["19-20","20-21","21-22","22-23","23-24"]
htn_r  = [2.48, 2.30, 3.60, 2.50, 3.25]
dm_r   = [0.90, 1.06, 1.36, 1.21, 1.44]
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(inter, htn_r, marker="o", color="0.15", lw=1.6, label="Hypertension")
ax.plot(inter, dm_r,  marker="s", color="0.55", lw=1.6, ls="--", label="Diabetes")
ax.set_xlabel("Transition interval"); ax.set_ylabel("Incidence rate (%)")
ax.set_ylim(0, 4); ax.legend(frameon=False)
save_fig(fig, "fig1_incidence_by_interval")

# Figure 2: baseline BMI category vs incidence.
htn = pd.read_parquet(os.path.join(DATA, "htn_analysis.parquet"))
htn["bmi_cat"] = pd.cut(htn["BMI"], [0,18.5,23,25,30,100],
                        labels=["Under","Normal","Over","Obese I","Obese II+"])
rates = htn.groupby("bmi_cat", observed=True)["incident"].mean() * 100
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.bar(range(len(rates)), rates.values, color="0.5", edgecolor="0.2", width=0.65)
ax.set_xticks(range(len(rates))); ax.set_xticklabels(rates.index)
ax.set_xlabel("Baseline BMI category"); ax.set_ylabel("HTN incidence rate (%)")
save_fig(fig, "fig2_bmi_incidence_gradient")

In [4]:
# Figure 3: Step 4 confounding adjustment (grouped bar).
stages = ["Crude", "PSM matched", "Overweight+"]
ach    = [3.53, 3.57, 4.07]
noach  = [3.05, 3.74, 3.95]
x = np.arange(len(stages)); w = 0.36
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.bar(x - w/2, ach,   w, color="0.25", edgecolor="0.1", label="BMI reduction achieved")
ax.bar(x + w/2, noach, w, color="0.7",  edgecolor="0.3", label="Not achieved")
ax.set_xticks(x); ax.set_xticklabels(stages)
ax.set_ylabel("t2 HTN incidence rate (%)"); ax.set_ylim(0, 5)
ax.legend(frameon=False)
save_fig(fig, "fig3_step4_confounding_adjustment")

# Figure 4: sensitivity forest of adjusted ORs.
labels = tab4["Threshold"].tolist()
OR = tab4["OR"].values; lo = tab4["CI_low"].values; hi = tab4["CI_high"].values
yy = np.arange(len(labels))[::-1]
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.errorbar(OR, yy, xerr=[OR - lo, hi - OR], fmt="o", color="0.15",
            ecolor="0.5", capsize=3, ms=5, lw=1.2)
ax.axvline(1.0, color="0.4", ls=":", lw=1)
ax.set_yticks(yy); ax.set_yticklabels(labels)
ax.set_xlabel("Adjusted OR (95% CI)"); ax.set_ylabel("BMI reduction threshold")
save_fig(fig, "fig4_sensitivity_forest")
print("All figures saved to results/figures (png + pdf, 600 dpi).")

All figures saved to results/figures (png + pdf, 600 dpi).
